In [1]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)
from Models.BERT_Model.BERT_Model import BERT_Lag
from Models.BERT_Model.train_BERT import train_loop , estimate_loss
from Models.Configs import BERTConfig, TrainConfig
from Datasets.DataLoader import CombinedBinDataLoader

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Mounted at /content/gdrive
Device set to cuda


In [2]:
model_config = BERTConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = BERT_Lag(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


In [3]:
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

Initialized loader with 57,983 chunks of size 16385.
Initialized loader with 3,052 chunks of size 16385.


In [4]:
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config)

Trainable parameters: 124,083,456


W0422 18:35:11.671000 3902 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Step    0 | Val FWD: 11.0079 | PPL_FWD: 60346.62 | Val BWD: 11.0028| PPL_BWD: 60042.37
step    0/3623 | tokens 262,144 | Train loss 11.0003 | lr 2.00e-06 | Time Since Last Train Print 80.8641 seconds


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


step   10/3623 | tokens 2,883,584 | Train loss 10.4118 | lr 2.20e-05 | Time Since Last Train Print 132.6111 seconds
step   20/3623 | tokens 5,505,024 | Train loss 9.5707 | lr 4.20e-05 | Time Since Last Train Print 124.3576 seconds
step   30/3623 | tokens 8,126,464 | Train loss 9.1412 | lr 6.20e-05 | Time Since Last Train Print 123.1232 seconds
step   40/3623 | tokens 10,747,904 | Train loss 8.6714 | lr 8.20e-05 | Time Since Last Train Print 121.7212 seconds
Step  50| Val FWD: 8.1721 | PPL_FWD: 3540.66 | Val BWD: 8.1657| PPL_BWD: 3518.13
step   50/3623 | tokens 13,369,344 | Train loss 8.2447 | lr 1.02e-04 | Time Since Last Train Print 136.1204 seconds
step   60/3623 | tokens 15,990,784 | Train loss 7.8505 | lr 1.22e-04 | Time Since Last Train Print 119.9309 seconds


KeyboardInterrupt: 

In [ ]:
loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, grad_acc_factor)
print(f"Val FWD: {loss_fwd:.4f} | PPL FWD: {ppl_fwd:.2f} | PPL BWD: {ppl_bwd:.2f} | Val BWD: {-100:.4f}")

In [ ]:
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_BERT_6_layers_combined_loss.pt"
torch.save({
            "step": 2770,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict()
        }, path)

In [ ]:
import tiktoken

def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=50, device=device):
    enc = tiktoken.get_encoding("gpt2")
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)  # [1, T]

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -model.config.block_size:]

            B, N = x_cond.size()
            pos = torch.arange(0, N, dtype=torch.long, device=device)

            with torch.amp.autocast('cuda'):
                h = model.transformer.drop(
                    model.transformer.wte(x_cond) + model.transformer.wpe(pos)
                )
                for block in model.transformer.h:
                    h = block(h, mask=True)
                h = model.transformer.ln_f(h)
                logits = model.lm_head(h)  # [1, T, vocab_size]

            logits = logits[:, -1, :].float() / temperature  # [1, vocab_size]

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)

    model.train()
    return enc.decode(x[0].tolist())

print(generate(model, """Ancient Rome (753 BC–AD 476) evolved from a small Italian city-state into a massive Mediterranean empire, structured into three main periods: the Monarchy/Kingdom (753–509 BC), the Republic (509–27 BC), and the Empire (27 BC–AD 476).
It became a dominant power under emperors like Augustus, falling in the West due to internal instability and external pressures, while the Eastern Byzantine Empire continued until 1453.
""", max_new_tokens=100, temperature=1, top_k=50))